# What does a forecast buy a home battery, and on which tariff?

One continuous MILP, driven by a Prophet forecast, over a 10 kWh / 1.5 kW battery on
30 Ausgrid households and a full simulated year — measured against the controllers a
household could actually install, on two price signals that reward completely different
behaviour.

### The households

30 sites, one per cluster: the centroid-nearest member of each of 30 k-means clusters over
all 300 Ausgrid households. They span the consumption shapes present in the population
rather than being a convenience sample, and `hems_study.study_units()` derives them from
the clustering rather than restating a list, so the rule is executable.

### The tariff axis

| | |
|---|---|
| **AU** | Ausgrid EA025 time-of-use on `Australia/Sydney`, working-day aware, spot-linked. Energy and a standing charge; **no capacity charge**. |
| **SI** | GEN-I Dinamični under `si_samooskrba`. Energy, a standing charge, **and an excess-power charge** measured per network block against a *dogovorjena obračunska moč* that is re-agreed each month from the peaks the controller itself realised the month before. |

The contract is therefore endogenous on SI: a controller that shaves its way onto a lower
agreed power is then held to it, and one that charges at night pays for the peak it made.
`Environment.converge_agreed_power` iterates each run to that fixed point.

### The controller axis

Every controller decides one interval at a time from what it can know, and all of them are
executed on the same battery and priced by the same evaluator — they differ *only* in how a
setpoint is chosen.

| | |
|---|---|
| `no_battery` | the reference, run as an idle policy so it carries every column the others do |
| `self_consumption` | what a residential inverter ships with: soak the surplus, cover the deficit, never trade |
| `fixed_schedule`, `delayed_pv_charge` | a clock, and feed-in damping |
| `price_threshold`, `price_rank_daily`, `tariff_arbitrage` | trade on the published day-ahead price |
| `peak_shaving`, `self_consumption_peak_shaving` | watch the meter (SI only — there is no capacity charge on AU for them to earn from) |
| `price_oracle` | `price_threshold` reading forward instead of backward. **Not deployable**; it bounds what foresight is worth to a threshold rule |
| `prophet` | the MILP on a Prophet forecast — the study's subject |
| `oracle` | the same MILP on realised data: perfect foresight *within the control horizon* |

### What makes the comparison fair

One battery (`align_envelope`), one environment, one settlement. The MILP used to solve its
own copy of the battery while the rules read the environment's, which differ by `eff²` — a
5.3 % larger charge rating and a 5.0 % smaller discharge rating — and the two were priced by
different functions. Both are now shared, and `Cost_EUR_Closed` values any energy left in the
pack at the end at the mean delivered import rate, so a rule that runs the battery flat in
December cannot book the difference as a saving.

In [1]:
### Imports
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import hems_study as hs
hs = importlib.reload(hs)

print(f"hems_study loaded: {len(hs.STUDY_ARMS)} arms, "
      f"{len(hs.rule_roster('SI')) + 1} controllers on SI, "
      f"{len(hs.rule_roster('AU')) + 1} on AU.")

hems_study loaded: 11 arms, 10 controllers on SI, 8 on AU.


/Users/summerscholl/Library/Python/3.14/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

Everything the study computes lives in `hems_study.py`; this notebook configures it, reads
what it wrote and draws the figures. `RUN_SWEEP = True` recomputes the lot — hours — and by
default the notebook loads the CSVs the sweep already produced. Work is checkpointed per
(arm, household), and a checkpoint carries the configuration it was produced under, so a run
under superseded rules is dropped rather than resumed into.

In [2]:
UNITS = hs.study_units()               # household -> the cluster it represents
ARMS = list(hs.ARM_ORDER)
RESULTS_DIR = hs.RESULTS_DIR
RUN_SWEEP = True                      # True -> recompute everything, resumable
N_SIM = 365
# Households in parallel, one process each. The sweep is 330 independent
# checkpointed runs; the household is the unit because both caches -- Prophet's
# table and the forecast-blind oracle -- are keyed per household, so one worker
# owning a household means full cache reuse and no races. N_JOBS = 1 is the old
# serial behaviour.
N_JOBS = 10

print(f"households    : {len(UNITS)}, one per k-means cluster over all 300 Ausgrid sites")
print(f"battery       : 10 kWh nameplate, 7.0 kWh usable (SOC 10-80 %), 1.5 kW AC, eff 0.95")
print(f"horizon       : 24 h (48 steps) and 11 h (22 steps) at 30-minute resolution")
print(f"simulation    : {N_SIM} days after 730 days of training data")
print(f"solver        : {hs.SOLVER_NAME}"
      f"{' (in-process; no subprocess per solve)' if hs.SOLVER_NAME == 'HiGHS' else ''}")
print(f"parallelism   : {N_JOBS} worker(s) over {len(UNITS)} households")

print(f"\narms          : {len(ARMS)}")
for arm in hs.STUDY_ARMS:
    spec = {k: v for k, v in arm.items() if k != "name"}
    print(f"   {arm['name']:18s} {spec}")

print(f"\ncontrollers")
for tariff in ("AU", "SI"):
    names = ["no_battery"] + [p.name for p in hs.rule_roster(tariff)] + ["prophet", "oracle"]
    print(f"   {tariff}: {', '.join(names)}")

print(f"\nreference arms: {hs.REFERENCE_ARM}")
print("The two peak-shaving rules run on SI only: Ausgrid EA025 as modelled carries no")
print("capacity charge, so on AU they could only spend round-trip losses.")

households    : 30, one per k-means cluster over all 300 Ausgrid sites
battery       : 10 kWh nameplate, 7.0 kWh usable (SOC 10-80 %), 1.5 kW AC, eff 0.95
horizon       : 24 h (48 steps) and 11 h (22 steps) at 30-minute resolution
simulation    : 365 days after 730 days of training data

arms          : 11
   AU_H24             {'tariff': 'AU', 'control_horizon': 48}
   AU_H11             {'tariff': 'AU', 'control_horizon': 22}
   SI_H24             {'tariff': 'SI', 'control_horizon': 48}
   SI_H11             {'tariff': 'SI', 'control_horizon': 22}
   AU_H24_leaked      {'tariff': 'AU', 'control_horizon': 48, 'leak_current_interval': True}
   AU_H24_persist     {'tariff': 'AU', 'control_horizon': 48, 'forecaster_kind': 'persistence'}
   SI_H24_persist     {'tariff': 'SI', 'control_horizon': 48, 'forecaster_kind': 'persistence'}
   AU_H24_pvnaive     {'tariff': 'AU', 'control_horizon': 48, 'forecaster_kind': 'pvnaive'}
   SI_H24_pvnaive     {'tariff': 'SI', 'control_horizon': 48, 'fore

## 2. Run or load

One checkpoint per (arm, household). Only households that finished every arm enter the
comparison, so a difference between arms is not a difference in sample.

In [3]:
if RUN_SWEEP:
    hs.run_arms(
        data_dir=str(Path("..") / "Input data" / "Ausgrid"),
        output_root=str(RESULTS_DIR),
        dataset_ids=hs.dataset_ids(),
        n_sim=N_SIM,
        n_jobs=N_JOBS,
    )

df_all = hs.collect_results(RESULTS_DIR)
if df_all.empty:
    raise SystemExit(
        f"No results in {RESULTS_DIR}.\n"
        f"Run  python hems_study.py  (hours, resumable), or set RUN_SWEEP = True above."
    )

print(f"{'arm':20s}{'tariff':>8s}{'households':>12s}{'with roster':>13s}")
for arm in sorted(df_all["arm"].unique(), key=hs.ARM_ORDER.index):
    sub = df_all[df_all["arm"] == arm]
    print(f"{arm:20s}{str(sub['tariff'].iloc[0]):>8s}{len(sub):>12d}"
          f"{int(sub['roster_complete'].sum()):>13d}")

# A checkpoint written before the rule roster existed carries cost_prophet and
# cost_oracle but none of the rules. Reading it as though it did is how a partial
# panel gets reported as a whole one, so those rows are held out and said so.
stale = df_all[~df_all["roster_complete"]]
if len(stale):
    print(f"\nHeld out -- no rule-based controllers on these runs, so they predate the "
          f"roster and cannot be compared against it: {len(stale)} run(s) across "
          f"{', '.join(sorted(stale['arm'].unique()))}")
    print("Re-run them with RUN_SWEEP = True to bring them onto the current comparison.")
df_all = df_all[df_all["roster_complete"]].reset_index(drop=True)
if df_all.empty:
    raise SystemExit(
        "No run carries the rule-based controllers yet, so there is nothing to compare "
        "the MILP against. Run the sweep: python hems_study.py"
    )

# A household enters only if it finished every arm still standing, so a
# difference between arms is not a difference in sample.
arms_present = sorted(df_all["arm"].unique(), key=hs.ARM_ORDER.index)
done = df_all.groupby("dataset")["arm"].nunique()
balanced = set(done[done == len(arms_present)].index)
held = set(df_all["dataset"]) - balanced
if held:
    print(f"\nHeld out to keep the panel balanced (finished some arms, not all): "
          f"{len(held)} -- {', '.join(sorted(held))}")
df_all = df_all[df_all["dataset"].isin(balanced)].reset_index(drop=True)

# A saving PERCENTAGE needs a positive baseline. A net exporter has
# cost_no_battery <= 0, and dividing by it flips the sign, so a site that saves
# money would read as one that loses it.
neg = df_all.loc[df_all["cost_no_battery"] <= 1e-9, "dataset"].unique()
if len(neg):
    print(f"\n! {len(neg)} site(s) have a non-positive no-battery cost; their euro "
          f"columns are valid but every _pct column is NaN: {', '.join(neg)}")

print(f"\nComparing {df_all['dataset'].nunique()} households across "
      f"{len(arms_present)} arms = {len(df_all)} runs.")


######################################################################
### ARM AU_H24: {'tariff': 'AU', 'control_horizon': 48}
######################################################################
30 datasets detected


########## [1/30] Ausgrid 138.csv ##########

=== Dataset: Ausgrid 138 ===
Native granularity (30 min): 52608 steps
Training: 35040 steps (730 days)
Simulation: 17520 steps (365 days)
Tariff: AU | import rate 0.0389..0.4740 EUR/kWh
  [checkpoint] already complete under this configuration; skipping


########## [2/30] Ausgrid 127.csv ##########

=== Dataset: Ausgrid 127 ===
Native granularity (30 min): 52608 steps
Training: 35040 steps (730 days)
Simulation: 17520 steps (365 days)
Tariff: AU | import rate 0.0389..0.4740 EUR/kWh
  [checkpoint] already complete under this configuration; skipping


########## [3/30] Ausgrid 65.csv ##########

=== Dataset: Ausgrid 65 ===
Native granularity (30 min): 52608 steps
Training: 35040 steps (730 days)
Simulation: 17520 steps (365 

KeyboardInterrupt: 

## 3. The comparison table

`saving` is against the same household with no battery, in the arm's own currency (AUD on
AU, EUR on SI) — read percentages *down* a tariff, never across one.

`gain_share_pct` is the share of the oracle's saving a controller captured. The oracle is
perfect foresight **within the control horizon**, not a whole-year optimum, so it is a
ceiling on this class of controller rather than the true optimum.

In [ ]:
long = hs.summarize(df_all)

# One row per (arm, controller): the median household, and the spread across them.
summary = (long.groupby(["arm", "controller"])
           .agg(Households=("cost", "size"),
                Cost=("cost", "median"),
                Saving=("saving", "median"),
                Saving_pct=("saving_pct", "median"),
                Gain_share_pct=("gain_share_pct", "median"),
                Worst_pct=("saving_pct", "min"),
                Best_pct=("saving_pct", "max"),
                EFC=("efc", "median"))
           .reset_index())
order = {c: i for i, c in enumerate(hs.controller_columns(df_all))}
summary = (summary.assign(_a=summary["arm"].map(hs.ARM_ORDER.index),
                          _c=summary["controller"].map(order))
           .sort_values(["_a", "_c"]).drop(columns=["_a", "_c"])
           .set_index(["arm", "controller"]))

summary.round(2)

In [ ]:
### The best deployable rule on each tariff, and what it costs to use it instead of the MILP
for tariff, arm in hs.REFERENCE_ARM.items():
    if arm not in set(long["arm"]):
        continue
    winner = hs.best_rule(long, arm)
    if winner is None:
        print(f"{tariff}: no rule-based controller has run on {arm} yet.")
        continue
    tab = summary.loc[arm]
    print(f"{tariff}  (reference arm {arm})")
    print(f"   best deployable rule : {winner.replace('_', ' ')}  "
          f"{tab.loc[winner, 'Cost']:.2f} vs MILP+Prophet {tab.loc['prophet', 'Cost']:.2f}")
    print(f"   share of oracle gain : rule {tab.loc[winner, 'Gain_share_pct']:.1f} %, "
          f"MILP+Prophet {tab.loc['prophet', 'Gain_share_pct']:.1f} %")
    print(f"   cycles per year      : rule {tab.loc[winner, 'EFC']:.1f}, "
          f"MILP+Prophet {tab.loc['prophet', 'EFC']:.1f}")
    if tab.loc[winner, "Cost"] < tab.loc["prophet", "Cost"]:
        print("   -> the forecast-free rule is CHEAPER than the forecast-driven MILP on")
        print("      the median household. Cell 15 says whether that is significant.")
    print()

## 4. Plots

Colour is the tariff — AU blue, SI orange — and within a tariff the controller family is a
shade of the same hue plus a marker shape, never a new colour. Every figure goes through
`Plotting_Functions`, which decides style, size and export, and writes the caption and the
provenance of each figure beside it under `Results/Figures/hems/`.

In [ ]:
### Shared chart styling
import importlib
import Plotting_Functions as pf

# `import` returns whatever the kernel cached the first time. Editing
# Plotting_Functions and re-running this cell would otherwise keep the old
# module, which shows up as a TypeError about an argument that does exist.
pf = importlib.reload(pf)

# titles=False: an IEEE figure carries no title of its own -- the caption below
# it, set in the article, is the title. The text is not lost; it goes into each
# export's SOURCE.md as the caption to use.
pf.use(subdir="hems", titles=False)
from Plotting_Functions import INK, INK_2, MUTED, SURFACE, SERIES, finish

# What a colour MEANS is this study's semantics, so it stays here; only the
# palette itself is shared.
TARIFF_COLOR = {"AU": SERIES[0], "SI": SERIES[1]}

# Controllers by family, so a reader can tell at a glance what KIND of thing a
# row is without memorising ten names.
FAMILY = {
    "no_battery": "reference",
    "self_consumption": "pv", "delayed_pv_charge": "pv",
    "fixed_schedule": "clock",
    "price_threshold": "price", "price_rank_daily": "price",
    "tariff_arbitrage": "price",
    "peak_shaving": "meter", "self_consumption_peak_shaving": "meter",
    "price_oracle": "diagnostic", "oracle": "diagnostic",
    "prophet": "milp",
}
FAMILY_SHADE = {"reference": 0.75, "pv": 0.55, "clock": 0.40, "price": 0.22,
                "meter": 0.05, "milp": 0.0, "diagnostic": 0.62}
FAMILY_MARKER = {"reference": "x", "pv": "o", "clock": "s", "price": "^",
                 "meter": "D", "milp": "*", "diagnostic": "v"}
LABEL = {
    "no_battery": "no battery", "self_consumption": "self-consumption",
    "fixed_schedule": "fixed schedule", "delayed_pv_charge": "delayed PV charge",
    "price_threshold": "price threshold", "price_rank_daily": "day-ahead price rank",
    "tariff_arbitrage": "tariff arbitrage", "peak_shaving": "peak shaving",
    "self_consumption_peak_shaving": "self-consumption + peak shaving",
    "price_oracle": "price threshold (foresight)", "prophet": "MILP + Prophet",
    "oracle": "MILP, perfect foresight",
}


def ctrl_color(tariff, controller):
    """Fill for one (tariff, controller): the tariff's hue, shaded by family."""
    return pf.mix(TARIFF_COLOR[tariff], "#ffffff",
                  FAMILY_SHADE[FAMILY.get(controller, "price")])


def ctrl_marker(controller):
    return FAMILY_MARKER[FAMILY.get(controller, "price")]


SAMPLE = f"{df_all['dataset'].nunique()} households, one per k-means cluster"
print(f"Style ready: {SAMPLE}, {len(arms_present)} arms.")

In [ ]:
### Every controller on the reference arm: one dot per household, the bar is the median
# The "comparison of all" figure, one per tariff -- the tariffs reward different
# behaviour, so the winner is not assumed to be the same on both.
rng = np.random.default_rng(0)

for tariff, arm in hs.REFERENCE_ARM.items():
    if arm not in set(long["arm"]):
        continue
    sub = long[long["arm"] == arm]
    ctrls = [c for c in hs.controller_columns(df_all) if c in set(sub["controller"])]
    ypos = np.arange(len(ctrls))[::-1]

    fig, ax = plt.subplots(figsize=pf.figsize(ratio=0.62))
    for y, c in zip(ypos, ctrls):
        vals = sub.loc[sub["controller"] == c, "saving_pct"].dropna()
        ax.scatter(vals, y + rng.uniform(-0.17, 0.17, len(vals)), s=30,
                   marker=ctrl_marker(c), color=ctrl_color(tariff, c),
                   alpha=0.6, lw=0.7, edgecolor=SURFACE, zorder=3)
        if len(vals):
            ax.scatter([vals.median()], [y], s=150, marker="|", color=INK, lw=2.0, zorder=4)
    ax.axvline(0, color=MUTED, lw=1.1, ls="--")
    ax.set_yticks(ypos, [LABEL.get(c, c) for c in ctrls])
    ax.grid(axis="y", visible=False)
    finish(ax, title=f"Every controller on {tariff}, {SAMPLE}",
           xlabel="Saving vs the same household with no battery [% of its energy bill]")
    pf.show(fig, f"controller_ranking_{tariff.lower()}")

In [ ]:
### What the forecast channel is worth: naive PV, Prophet PV, perfect PV
# Prophet's consumption model beats seasonal-naive; its generation model loses to
# it. A single "with forecast" arm averages those two facts into one number, so
# these arms hold consumption at Prophet and vary only the roof.
CHANNEL = {"pvnaive": "PV: yesterday", "prophet": "PV: Prophet", "pvtruth": "PV: perfect",
           "persistence": "both: yesterday"}
kinds = [k for k in ("persistence", "pvnaive", "prophet", "pvtruth")
         if k in set(df_all["forecaster_kind"])]

if len(kinds) > 1:
    fig, ax = plt.subplots(figsize=pf.figsize(ratio=0.5))
    width = 0.38
    for i, tariff in enumerate(sorted(set(df_all["tariff"]))):
        sel = df_all[(df_all["tariff"] == tariff) & (df_all["control_horizon"] == 48)]
        vals = [sel.loc[sel["forecaster_kind"] == k, "cost_prophet"].median() -
                sel.loc[sel["forecaster_kind"] == k, "cost_oracle"].median()
                for k in kinds]
        x = np.arange(len(kinds)) + (i - 0.5) * width
        ax.bar(x, vals, width=width * 0.9, color=TARIFF_COLOR[tariff], label=tariff)
        for xi, v in zip(x, vals):
            if v == v:
                ax.annotate(f"{v:.2f}", xy=(xi, v), xytext=(0, 3),
                            textcoords="offset points", ha="center",
                            color=INK_2, fontsize=8)
    ax.set_xticks(np.arange(len(kinds)), [CHANNEL[k] for k in kinds])
    ax.grid(axis="x", visible=False)
    finish(ax, title="What better knowledge of the roof is worth",
           ylabel="Regret against perfect foresight [median, per household-year]")
    pf.key_legend(ax, [plt.Line2D([], [], marker="s", ls="", ms=9,
                                  color=TARIFF_COLOR[t])
                       for t in sorted(set(df_all["tariff"]))],
                  where="upper right", labels=sorted(set(df_all["tariff"])), title="Tariff")
    pf.show(fig, "forecast_channel_regret")
else:
    print(f"Only the {kinds} arm has run; the forecast-channel figure needs the "
          f"pvnaive/pvtruth arms. Run them with RUN_SWEEP = True.")

In [ ]:
### Cycles against saving: what a saving costs in pack life
# A controller high on the y axis and far right on the x axis is buying its
# saving out of the battery, not out of the tariff. At 250 EUR/kWh over a
# 6000-cycle life a 10 kWh pack prices one equivalent full cycle at 0.417 EUR.
import Battery_Economics as be

for tariff, arm in hs.REFERENCE_ARM.items():
    sub = long[(long["arm"] == arm) & (long["controller"] != "no_battery")]
    if sub.empty or sub["efc"].isna().all():
        continue
    med = sub.groupby("controller").agg(efc=("efc", "median"),
                                        saving=("saving", "median")).dropna()

    fig, ax = plt.subplots(figsize=pf.figsize(ratio=0.58))
    for c, row in med.iterrows():
        ax.scatter([row["efc"]], [row["saving"]], s=130, marker=ctrl_marker(c),
                   color=ctrl_color(tariff, c), edgecolor=SURFACE, lw=1.2, zorder=3)
        ax.annotate(LABEL.get(c, c), xy=(row["efc"], row["saving"]),
                    xytext=(8, 4), textcoords="offset points",
                    color=INK, fontsize=8.5, zorder=5)
    # What the cycles on the x axis cost, at the pack price the economics module
    # carries. Anything below this line spends more on wear than it saves.
    xs = np.linspace(0, med["efc"].max() * 1.15, 50)
    ax.plot(xs, xs * be.cycle_cost_eur_per_efc(10.0), color=MUTED, lw=1.2, ls="--",
            zorder=2)
    ax.annotate("wear cost at 250 EUR/kWh, 6000 EFC",
                xy=(xs[-1], xs[-1] * be.cycle_cost_eur_per_efc(10.0)),
                xytext=(-4, 6), textcoords="offset points", ha="right",
                color=MUTED, fontsize=8)
    finish(ax, title=f"What a saving costs in pack life, {tariff}",
           xlabel="Equivalent full cycles [median household]",
           ylabel="Saving vs no battery")
    pf.show(fig, f"wear_vs_saving_{tariff.lower()}")

## 5. Read-out

The households are **paired** — every one is run under every arm — so the per-site
difference is the unit of analysis, and a box plot of two independent-looking
distributions understates the evidence. Wilcoxon signed-rank rather than a t-test: 30 sites
of cost differences are not plausibly normal and a few of them sit far out.

In [ ]:
### Every controller against MILP+Prophet, household by household
for tariff, arm in hs.REFERENCE_ARM.items():
    if arm not in set(long["arm"]):
        continue
    rep = hs.paired_controllers(long, arm, reference="prophet")
    if rep.empty:
        continue
    print(f"=== {tariff} ({arm}): controller - MILP+Prophet, per household ===")
    print(f"{'comparison':38s}{'median':>9s}{'IQR':>18s}{'better':>8s}{'worse':>7s}{'p':>10s}")
    for _, r in rep.iterrows():
        name = r['comparison'].split(' - ')[0]
        p = "n/a" if r["p_value"] != r["p_value"] else f"{r['p_value']:.4f}"
        print(f"{LABEL.get(name, name):38s}{r['median_diff']:9.2f}"
              f"{f'[{r.q1_diff:.2f}, {r.q3_diff:.2f}]':>18s}"
              f"{r['n_b_greater']:>8d}{r['n_a_greater']:>7d}{p:>10s}")
    print("  negative median = the controller is CHEAPER than MILP+Prophet")
    print()

### And every arm against its tariff's reference arm
for tariff in sorted(set(df_all["tariff"])):
    rep = hs.paired_arms(df_all, tariff, metric="cost_prophet")
    if rep.empty:
        continue
    print(f"=== {tariff}: arm - {hs.REFERENCE_ARM[tariff]}, per household ===")
    for _, r in rep.iterrows():
        p = "n/a" if r["p_value"] != r["p_value"] else f"{r['p_value']:.4f}"
        print(f"  {r['comparison']:44s}{r['median_diff']:9.2f}  p={p}")
    print()

### Caveats

* **The "oracle" is not the optimum.** It is the same receding-horizon controller reading
  realised data, so it is perfect foresight *within the control horizon* — 24 h or 11 h, not
  a year. Every `gain_share_pct` divides by it, so it bounds this class of controller rather
  than what a battery could do. A whole-period solve is the honest denominator and is not
  in this notebook yet.
* **A percentage on AU is not a percentage on SI.** The two are different currencies over
  different bills; `saving` in the arm's own units settles comparisons across tariffs, and
  the percentages are only comparable down a tariff.
* **The saving is against the energy bill, not the invoice.** The standing charge is
  reported as `Fixed_EUR` and excluded from `Cost_EUR`, because no controller can move it.
  It is not small: 705 EUR/a against a 566 EUR/a energy bill on Ausgrid 127, so a "40 %
  saving" is closer to 18 % of what the household actually pays.
* **The MILP optimises energy only.** On SI it is then billed an excess-power charge it
  never saw in its objective, which is why a peak-shaving rule can beat it there. That is a
  statement about the objective, not about MILP.
* **Every controller still has perfect foresight of the price** inside its horizon. On AU
  that is close to true (pre-dispatch is published); it is the load and the roof that are
  forecast.
* **Which channel Prophet is good at varies by household**, and the study should not claim
  otherwise until the full panel is in. On Ausgrid 127 the PV model scores −0.13 skill
  against seasonal-naive while consumption scores +0.21; on Ausgrid 148 it is the other way
  round (+0.14 / −0.15). That variation is the reason the `*_pvnaive` and `*_pvtruth` arms
  exist — a single forecast arm averages the two channels into one number and hides which
  one is carrying the result.
* **Wear is reported, not optimised.** `Equivalent_Full_Cycles` and the wear cost are
  reported for every controller, but no controller has a wear price in its objective by
  default. Turning it on (`cycle_cost_eur_per_efc`) changes what the MILP decides, and
  deliberately stops its objective being the reported bill.
* **`price_oracle` is a diagnostic.** It reads the whole year and is never a
  recommendation; it exists to split the gap to the MILP into "the rule is too simple" and
  "the rule cannot see the future".
* **One battery size.** 10 kWh nameplate at 1.5 kW throughout. The capacity sweep
  (2.5–30 kWh) is not in this notebook yet.